# 06 — Perbandingan BiLSTM vs IndoBERT dan Ekspor Laporan

Notebook ini menggabungkan hasil dari notebook 02–05:

- baseline BiLSTM,
- BiLSTM setelah HPO,
- baseline IndoBERT,
- IndoBERT setelah HPO.

Yang dilakukan:

1. Tabel metrik holdout + bootstrap CI.
2. Kurva ROC/PR gabungan.
3. Uji McNemar dan paired bootstrap AUC.
4. Analisis error (false positive/negative).
5. Ekspor tabel + ringkasan Markdown untuk paper.

## Prinsip perbandingan

Semua model harus dievaluasi pada **holdout yang sama**. Jangan membandingkan
angka dari split berbeda. Selisih point estimate saja tidak cukup; sertakan
interval kepercayaan dan uji berpasangan.

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Notebook biasanya dijalankan dari folder notebooks/. Kita pindah ke root
# project agar semua path artefak konsisten.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "artifacts" / "data"
BERT_DIR = PROJECT_ROOT / "artifacts" / "bert" / "input"
RUNS_DIR = PROJECT_ROOT / "artifacts" / "notebook_runs"
for path in (DATA_DIR, BERT_DIR, RUNS_DIR):
    path.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Data dir     :", DATA_DIR)
print("BERT input   :", BERT_DIR)
print("Runs dir     :", RUNS_DIR)

In [ ]:
from app.experiment.reporting import (
    bootstrap_ci,
    classification_metrics,
    compare_two_models,
    load_probabilities,
    plot_roc_pr,
)

pd.set_option("display.max_colwidth", 200)

In [ ]:
RUN_SPECS = {
    "BiLSTM baseline": RUNS_DIR / "bilstm_baseline" / "probs_holdout.csv",
    "BiLSTM tuned": RUNS_DIR / "bilstm_tuned" / "probs_holdout.csv",
    "IndoBERT baseline": RUNS_DIR / "bert_baseline" / "probs_holdout.csv",
    "IndoBERT tuned": RUNS_DIR / "bert_tuned" / "probs_holdout.csv",
}

available = {name: path for name, path in RUN_SPECS.items() if path.exists()}
missing = [name for name in RUN_SPECS if name not in available]
print("Tersedia:", list(available))
print("Belum ada:", missing)
if not available:
    raise RuntimeError("Belum ada hasil model. Jalankan notebook 02–05 terlebih dahulu.")

## Tabel metrik holdout + 95% CI

Setiap baris adalah satu model. CI dihitung dengan stratified bootstrap
(2.000 resampling).

In [ ]:
rows = []
probability_sets = {}
for name, path in available.items():
    y_true, y_prob, _ = load_probabilities(path)
    probability_sets[name] = (y_true, y_prob)
    metrics = classification_metrics(y_true, y_prob)
    ci = bootstrap_ci(y_true, y_prob, n_bootstrap=2000, seed=42)
    ci_map = {row["metric"]: (row["lo"], row["hi"]) for _, row in ci.iterrows()}
    rows.append({
        "model": name,
        "n": int(metrics["n"]),
        "accuracy": metrics["accuracy"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "roc_auc": metrics["roc_auc"],
        "f1_ci": f"{ci_map['f1'][0]:.3f}–{ci_map['f1'][1]:.3f}",
        "auc_ci": f"{ci_map['roc_auc'][0]:.3f}–{ci_map['roc_auc'][1]:.3f}",
    })

comparison_table = pd.DataFrame(rows)
comparison_table

In [ ]:
if probability_sets:
    plot_roc_pr(probability_sets, RUNS_DIR / "comparison_roc_pr.png", title="Perbandingan model pada frozen holdout")
    print("Plot tersimpan di", RUNS_DIR / "comparison_roc_pr.png")

## Uji statistik berpasangan

- **McNemar**: apakah pola benar/salah dua model berbeda secara signifikan?
- **Paired bootstrap AUC**: selisih ROC-AUC + CI + p-value.

Keduanya memakai sampel holdout yang sama.

In [ ]:
names = list(probability_sets)
statistical_rows = []
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        name_a, name_b = names[i], names[j]
        y_a, p_a = probability_sets[name_a]
        y_b, p_b = probability_sets[name_b]
        comparison = compare_two_models(
            y_a, p_a, p_b,
            name_a=name_a, name_b=name_b,
            n_bootstrap=2000, seed=42,
        )
        statistical_rows.append({
            "model_a": name_a,
            "model_b": name_b,
            "delta_f1": comparison["delta"]["f1"],
            "delta_auc": comparison["delta"]["roc_auc"],
            "mcnemar_p": comparison["mcnemar"]["p_value"],
            "auc_diff": comparison["paired_bootstrap_auc"]["auc_a_minus_b"],
            "auc_diff_ci": (
                f"{comparison['paired_bootstrap_auc']['ci_lo']:.3f}–"
                f"{comparison['paired_bootstrap_auc']['ci_hi']:.3f}"
            ),
            "auc_diff_p": comparison["paired_bootstrap_auc"]["p_value"],
        })

pd.DataFrame(statistical_rows)

## Analisis error

Kita lihat contoh false positive (diprediksi unsafe padahal gold safe) dan
false negative (diprediksi safe padahal gold unsafe). Ini bahan diskusi yang
kuat di pembahasan paper.

In [ ]:
best_model_name = comparison_table.sort_values("f1", ascending=False).iloc[0]["model"]
y_true, y_prob = probability_sets[best_model_name]
pred = (y_prob >= 0.5).astype(int)
probs_df = pd.read_csv(available[best_model_name])

false_positives = probs_df[(y_true == 0) & (pred == 1)].copy()
false_negatives = probs_df[(y_true == 1) & (pred == 0)].copy()

print("Model:", best_model_name)
print("False positives:", len(false_positives))
print(false_positives.head(10).to_string(index=False))
print("\nFalse negatives:", len(false_negatives))
print(false_negatives.head(10).to_string(index=False))

## Ekspor untuk paper

Sel ini menulis:

- `comparison_holdout.csv` — tabel metrik,
- `statistical_tests.csv` — uji berpasangan,
- `comparison_summary.md` — ringkasan siap tempel.

In [ ]:
comparison_table.to_csv(RUNS_DIR / "comparison_holdout.csv", index=False)
pd.DataFrame(statistical_rows).to_csv(RUNS_DIR / "statistical_tests.csv", index=False)

best_row = comparison_table.sort_values("f1", ascending=False).iloc[0]
summary = f"""# Ringkasan Perbandingan Model

Holdout: frozen, n = {int(best_row['n'])}.

| Model | F1 | CI F1 | ROC-AUC | CI AUC |
|---|---:|---|---:|---|
"""
for _, row in comparison_table.iterrows():
    summary += f"| {row['model']} | {row['f1']:.3f} | {row['f1_ci']} | {row['roc_auc']:.3f} | {row['auc_ci']} |\n"

summary += f"""
Model dengan F1 tertinggi: **{best_row['model']}**.

Catatan: perbedaan antar model harus dibaca bersama uji McNemar dan paired
bootstrap AUC di `statistical_tests.csv`. Jangan mengklaim keunggulan hanya
dari selisih point estimate.
"""
(RUNS_DIR / "comparison_summary.md").write_text(summary, encoding="utf-8")
print(summary)

## Checklist laporan paper

- [ ] Search space HPO dicantumkan (parameter + rentang).
- [ ] Jumlah trial dan strategi disebutkan.
- [ ] Metrik seleksi validation disebutkan.
- [ ] Holdout tidak dipakai untuk tuning.
- [ ] Setiap metrik menyebut n + 95% CI.
- [ ] Perbandingan model memakai uji berpasangan.
- [ ] Ada ablasi synthetic vs real-only.
- [ ] Ada analisis error (false positive/negative).
- [ ] Seed dan varians antar run dilaporkan.
- [ ] Keterbatasan (n kecil, data sintetik, OCR noise) ditulis jujur.